# Customer Churn Predictor Notebook

This notebook contains a complete data analysis and modeling pipeline for a synthetic SaaS churn dataset. It includes dataset generation, cleaning, EDA, feature engineering, model training, evaluation, artifact saving, and README generation.

## 1. Import Required Libraries

Import standard libraries and machine learning utilities for the pipeline.

In [ ]:
import json
import logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
sns.set(style='whitegrid', palette='muted')
OUTPUT_DIR = Path('..') / 'outputs' if Path('..').exists() else Path('.') / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path('..') / 'data' if Path('..').exists() else Path('.') / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Dataset

Load the dataset from a local file when available, with a fallback to synthetic generation.

In [ ]:
def generate_synthetic_saas_data(n_samples=5000, churn_rate=0.15, random_state=42):
    rng = np.random.default_rng(random_state)
    tenure_months = rng.integers(1, 61, size=n_samples)
    monthly_charges = np.round(rng.normal(80, 22, size=n_samples).clip(25, 220), 2)
    support_tickets_raised = rng.poisson(0.6, size=n_samples)
    data_usage_gb = np.round(rng.normal(92, 32, size=n_samples).clip(5, 320), 1)
    churned = np.zeros(n_samples, dtype=int)
    churn_indices = rng.choice(n_samples, size=int(np.floor(n_samples * churn_rate)), replace=False)
    churned[churn_indices] = 1
    churn_mask = churned == 1
    if churn_mask.any():
        monthly_charges[churn_mask] = np.round(
            np.clip(monthly_charges[churn_mask] + rng.normal(18, 12, churn_mask.sum()), 35, 260),
            2,
        )
        support_tickets_raised[churn_mask] = np.clip(
            support_tickets_raised[churn_mask] + rng.integers(1, 4, churn_mask.sum()), 0, None
        )
        tenure_months[churn_mask] = np.clip(
            tenure_months[churn_mask] - rng.integers(4, 18, churn_mask.sum()), 1, None
        )
        data_usage_gb[churn_mask] = np.round(
            np.clip(data_usage_gb[churn_mask] + rng.normal(8, 18, churn_mask.sum()), 10, 360),
            1,
        )
    return pd.DataFrame({
        'tenure_months': tenure_months,
        'monthly_charges': monthly_charges,
        'support_tickets_raised': support_tickets_raised,
        'data_usage_gb': data_usage_gb,
        'churned': churned,
    })

def load_or_generate_dataset(path: Path):
    if path.exists():
        logging.info(f'Loading dataset from {path}')
        return pd.read_csv(path)
    logging.info(f'Dataset not found at {path}. Generating synthetic data.')
    df = generate_synthetic_saas_data()
    df.to_csv(path, index=False)
    logging.info(f'Synthetic dataset saved to {path}')
    return df

dataset_path = DATA_DIR / 'customer_churn.csv'
df = load_or_generate_dataset(dataset_path)
df.head()

## 3. Data Cleaning & Preprocessing

Handle missing values, duplicates, type conversion, and outliers.

In [ ]:
df = df.copy()
logging.info('Original shape: %s', df.shape)
df = df.drop_duplicates()
logging.info('After dropping duplicates: %s', df.shape)
df = df.fillna(df.median(numeric_only=True))
df['tenure_months'] = df['tenure_months'].astype(int)
df['support_tickets_raised'] = df['support_tickets_raised'].astype(int)
df['monthly_charges'] = df['monthly_charges'].clip(lower=20, upper=300)
df['data_usage_gb'] = df['data_usage_gb'].clip(lower=0, upper=400)
df.info()

## 4. Exploratory Data Analysis (EDA)

Generate summary statistics, correlation matrix, and distribution plots.

In [ ]:
display(df.describe())
display(df['churned'].value_counts(normalize=True).rename('ratio'))
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'notebook_correlation_matrix.png')
plt.show()
plt.figure(figsize=(10, 5))
sns.countplot(x='churned', data=df, palette='Set2')
plt.title('Churn Class Distribution')
plt.xticks([0, 1], ['Active', 'Churned'])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'notebook_churn_distribution.png')
plt.show()

## 5. Feature Engineering

Create derived features and build a preprocessing pipeline.

In [ ]:
df['support_ticket_rate'] = (df['support_tickets_raised'] / df['tenure_months']).replace([np.inf, -np.inf], 0).fillna(0)
numeric_features = ['tenure_months', 'monthly_charges', 'support_tickets_raised', 'data_usage_gb', 'support_ticket_rate']
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
])
df.head()

## 6. Train/Test Split & Scaling

Split the dataset and scale numeric features using the pipeline.

In [ ]:
X = df.drop(columns=['churned'])
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)
logging.info('Train shape: %s, Test shape: %s', X_train.shape, X_test.shape)

## 7. Model Training

Train logistic regression and random forest models, including hyperparameter search.

In [ ]:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_prepared, y_train)
logging.info('Balanced train class counts: %s', dict(pd.Series(y_train_balanced).value_counts()))
logistic_pipeline = Pipeline([
    ('clf', LogisticRegression(solver='liblinear', max_iter=1000, random_state=42))
])
rf_pipeline = Pipeline([
    ('clf', RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42))
])
logistic_grid = GridSearchCV(logistic_pipeline, {'clf__C': [0.1, 1.0, 5.0]}, cv=3, scoring='recall', n_jobs=-1, verbose=0)
rf_grid = GridSearchCV(rf_pipeline, {'clf__n_estimators': [100, 150], 'clf__max_depth': [6, 8]}, cv=3, scoring='recall', n_jobs=-1, verbose=0)
logistic_grid.fit(X_train_balanced, y_train_balanced)
rf_grid.fit(X_train_balanced, y_train_balanced)
logging.info('Logistic best params: %s', logistic_grid.best_params_)
logging.info('Random Forest best params: %s', rf_grid.best_params_)

## 8. Model Evaluation

Evaluate models using classification metrics and ROC curve visualization.

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=['active', 'churned'], digits=4)
    auc = roc_auc_score(y_test, y_score)
    print(f'--- {name} ---')
    print(f'ROC AUC: {auc:.4f}')
    print(report)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['active', 'churned'], yticklabels=['active', 'churned'])
    plt.title(f'Confusion Matrix - {name}')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'notebook_confusion_matrix_{name.replace(' ', '_').lower()}.png')
    plt.show()
    fpr, tpr, _ = roc_curve(y_test, y_score)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'ROC AUC = {auc:.4f}')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.title(f'ROC Curve - {name}')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'notebook_roc_curve_{name.replace(' ', '_').lower()}.png')
    plt.show()
    return {'model': name, 'roc_auc': float(auc), 'confusion_matrix': cm.tolist(), 'classification_report': report}

logistic_results = evaluate_model('Logistic Regression', logistic_grid.best_estimator_, X_test_prepared, y_test)
rf_results = evaluate_model('Random Forest', rf_grid.best_estimator_, X_test_prepared, y_test)
results = [logistic_results, rf_results]

## 9. Save Model & Results

Serialize the trained models and save evaluation metrics and predictions to disk.

In [ ]:
MODEL_DIR = Path('..') / 'models' if Path('..').exists() else Path('.') / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(logistic_grid.best_estimator_, MODEL_DIR / 'logistic_pipeline.joblib')
joblib.dump(rf_grid.best_estimator_, MODEL_DIR / 'random_forest_pipeline.joblib')
with open(OUTPUT_DIR / 'evaluation_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
predictions = X_test.copy()
predictions['actual_churn'] = y_test.values
predictions['predicted_churn_logistic'] = logistic_grid.best_estimator_.predict(X_test_prepared)
predictions['predicted_churn_rf'] = rf_grid.best_estimator_.predict(X_test_prepared)
predictions.to_csv(OUTPUT_DIR / 'predictions_sample.csv', index=False)
logging.info('Saved models and result artifacts')

## 10. Generate README.md

Automatically generate or update the README file describing the project and usage.

In [ ]:
readme_content = '''# Customer Churn Predictor

A complete end-to-end churn prediction project for a synthetic SaaS dataset.

## Features
- Synthetic SaaS churn dataset generation
- Data cleaning, EDA, and visualization
- Feature engineering and scaling
- Class balancing using SMOTE
- Logistic Regression and Random Forest modeling
- Evaluation with ROC-AUC, confusion matrix, and recall metrics
- Model serialization and output artifact saving

## Files
- notebooks/analysis.ipynb - interactive notebook for end-to-end pipeline
- customer_churn_pipeline.py - scripted pipeline implementation
- requirements.txt - Python dependencies
- outputs/ - saved evaluation plots and metrics
- models/ - serialized model objects

## Setup
1. Create a virtual environment: `python3 -m venv .venv`
2. Activate the environment: `source .venv/bin/activate`
3. Install dependencies: `pip install -r requirements.txt`

## Usage
- Run the notebook cells in order.
- Execute the script: `python customer_churn_pipeline.py`.

## Tools & Technologies
- Python 3.8+
- pandas, numpy, scikit-learn, imbalanced-learn
- matplotlib, seaborn
- joblib for model serialization
- VS Code and Git

## License
MIT License
'''
with open(Path('..') / 'README.md' if Path('..').exists() else Path('.') / 'README.md', 'w') as readme_file:
    readme_file.write(readme_content)
logging.info('README.md generated or updated.')

## 11. Tools & Technologies

Write a tools list file for the project.

In [ ]:
tools = [
    'Python 3.8+',
    'pandas',
    'numpy',
    'scikit-learn',
    'imbalanced-learn',
    'matplotlib',
    'seaborn',
    'joblib',
    'VS Code',
    'Git',
    'pytest (optional)',
]
with open(OUTPUT_DIR / 'tools_and_technologies.txt', 'w') as f:
    f.write('
'.join(tools))
logging.info('Tools and technologies documented in outputs/tools_and_technologies.txt')

## 12. Environment & Requirements

Generate curated requirements and export instructions.

In [ ]:
requirements = [
    'numpy',
    'pandas',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'imbalanced-learn',
    'joblib',
]
with open(Path('..') / 'requirements.txt' if Path('..').exists() else Path('.') / 'requirements.txt', 'w') as f:
    f.write('
'.join(requirements))
logging.info('requirements.txt written with curated package list.')
print('To export the environment, run: pip freeze > requirements.txt')